In [1]:
import numpy as np
import pandas as pd

### Challenge A

In [28]:
df1 = pd.read_csv('Dealers.csv')
df2 = pd.read_csv('Sales.csv')

In [29]:
df2['Sale_Date'] = pd.to_datetime(df2['Sale_Date'])
df2['Year_Month'] = df2['Sale_Date'].dt.to_period('M')

In [30]:
monthly_sales = df2.groupby(['Dealer_ID', 'Year_Month'])['Sale_ID'].count().reset_index(name='Bikes_Sold')

In [44]:
avg_monthly_sales = monthly_sales.groupby('Dealer_ID')['Bikes_Sold'].mean().reset_index()

In [45]:
avg_monthly_sales

,Dealer_ID,Bikes_Sold
0,DLR001,1.586207
1,DLR002,1.694915
2,DLR003,1.793651
3,DLR004,1.629630
4,DLR005,1.833333
...,...,...
195,DLR196,1.750000
196,DLR197,1.750000
197,DLR198,1.946429
198,DLR199,1.910714


In [32]:
merged_df = pd.merge(avg_monthly_sales, df1, on='Dealer_ID', how='inner')

In [33]:
merged_df['Completion_Rate'] = merged_df['Bikes_Sold'] / merged_df['Sales_Target']

In [34]:
top_regions = merged_df.groupby('Region')['Completion_Rate'].mean().sort_values(ascending=False)
top_formats = merged_df.groupby('Dealer_Type')['Completion_Rate'].mean().sort_values(ascending=False)

In [41]:
top_regions * 100

Region
South      1.681656
East       1.597663
West       1.563051
Central    1.531095
North      1.508495
Name: Completion_Rate, dtype: float64

In [40]:
top_formats * 100

Dealer_Type
Studio Store             2.897825
Authorized Dealership    1.555279
Flagship Showroom        0.897384
Name: Completion_Rate, dtype: float64

In [47]:
merged_df

,Dealer_ID,Bikes_Sold,Dealer_Name,Dealer_Type,State,City,Region,Opening_Year,Sales_Target,Dealer_Rating,Completion_Rate
0,DLR001,1.586207,Royal Enfield Flagship Store - Ujjain,Flagship Showroom,Madhya Pradesh,Ujjain,Central,2012,178,4.3,0.008911
1,DLR002,1.694915,Thrissur Royal Drive Royal Enfield,Authorized Dealership,Kerala,Thrissur,South,2007,155,4.2,0.010935
2,DLR003,1.793651,Royal Enfield Flagship Store - Salem,Flagship Showroom,Tamil Nadu,Salem,South,2021,227,4.2,0.007902
3,DLR004,1.629630,South Delhi Royal Drive Royal Enfield,Authorized Dealership,Delhi,South Delhi,North,2018,108,4.2,0.015089
4,DLR005,1.833333,Karnal Brand Store Royal Enfield,Authorized Dealership,Haryana,Karnal,North,2010,134,4.1,0.013682
...,...,...,...,...,...,...,...,...,...,...,...
195,DLR196,1.750000,Muzaffarpur Wheelz Royal Enfield,Authorized Dealership,Bihar,Muzaffarpur,East,2012,122,4.6,0.014344
196,DLR197,1.750000,Royal Enfield Flagship Store - Dibrugarh,Flagship Showroom,Assam,Dibrugarh,East,2009,219,4.4,0.007991
197,DLR198,1.946429,Royal Enfield Flagship Store - Tirupati,Flagship Showroom,Andhra Pradesh,Tirupati,South,2009,151,4.5,0.012890
198,DLR199,1.910714,Jalandhar Brand Store Royal Enfield,Authorized Dealership,Punjab,Jalandhar,North,2010,113,3.9,0.016909


### Challenge B

In [48]:
customers = pd.read_csv('Customers.csv')
finance = pd.read_csv('Finance.csv')
sales = pd.read_csv('Sales.csv')

In [49]:
finance_cust = pd.merge(finance, customers, on='Customer_ID', how='inner')

In [50]:
finance_cust['Annual_EMI'] = finance_cust['EMI'] * 12
finance_cust['EMI_Burden_Pct'] = (finance_cust['Annual_EMI'] / finance_cust['Annual_Income']) * 100

In [51]:
emi_burden = finance_cust.groupby('Age_Group')['EMI_Burden_Pct'].mean().sort_values(ascending=False)

In [52]:
emi_burden

Age_Group
60+      8.571174
18-25    8.226242
26-35    7.903274
36-45    7.779547
46-60    7.097771
Name: EMI_Burden_Pct, dtype: float64

In [53]:
interest_rates = finance_cust.groupby('Customer_Segment')['Interest_Rate_Pct'].mean().sort_values(ascending=False)

In [54]:
sales_cust = pd.merge(sales, customers, on='Customer_ID', how='inner')
finance_penetration = sales_cust.groupby('Customer_Segment')['Payment_Mode'].apply(lambda x: (x == 'Finance').mean() * 100).sort_values(ascending=False)

In [55]:
interest_rates

Customer_Segment
First-time Buyer       10.561830
Enthusiast             10.518907
Touring / Adventure    10.476221
Premium Collector      10.476045
Daily Commuter         10.459720
Name: Interest_Rate_Pct, dtype: float64

In [68]:
finance_penetration

Customer_Segment
Premium Collector      45.566413
Enthusiast             45.397715
Touring / Adventure    44.948689
Daily Commuter         44.797033
First-time Buyer       44.583333
Name: Payment_Mode, dtype: float64

### Challenge C

In [57]:
service = pd.read_csv('Service.csv')
sales = pd.read_csv('Sales.csv')
motorcycles = pd.read_csv('Motorcycles.csv')

In [58]:
sales_per_model = sales['Bike_ID'].value_counts().reset_index(name='Total_Sales')

In [59]:
paid_services = service[service['Service_Type'] == 'Paid Service']
paid_per_model = paid_services['Bike_ID'].value_counts().reset_index(name='Paid_Services')

In [60]:
retention = pd.merge(sales_per_model, paid_per_model, on='Bike_ID', how='left').fillna(0)
retention = pd.merge(retention, motorcycles[['Bike_ID', 'Model']], on='Bike_ID', how='left')

In [61]:
retention['Retention_Score'] = (retention['Paid_Services'] / retention['Total_Sales']) * 100

In [62]:
retention[['Model', 'Retention_Score']].sort_values(by='Retention_Score', ascending=False).head(5)

,Model,Retention_Score
13,Thunderbird 350X,21.238938
11,Shotgun 650,19.704433
4,Himalayan 450,16.180049
9,Super Meteor 650,15.942029
6,Guerrilla 450,15.897436


In [63]:
service_costs = service[service['Service_Type'].isin(['Periodic Maintenance', 'Breakdown Repair'])]

In [64]:
cost_summary = service_costs.groupby('Service_Type')[['Parts_Cost', 'Labor_Cost']].mean()

In [65]:
cost_summary['Parts_to_Labor_Ratio'] = cost_summary['Parts_Cost'] / cost_summary['Labor_Cost']
cost_summary

,Parts_Cost,Labor_Cost,Parts_to_Labor_Ratio
Service_Type,,,
Breakdown Repair,6480.939983,2411.125951,2.687931
Periodic Maintenance,2370.438310,1063.275592,2.229373


### Challenge D

In [70]:
claims = pd.read_csv('WarrantyClaims.csv')
sales = pd.read_csv('Sales.csv')
motorcycles = pd.read_csv('Motorcycles.csv')

In [71]:
sales_moto = pd.merge(sales, motorcycles, on='Bike_ID')
claims_moto = pd.merge(claims, motorcycles, on='Bike_ID')

In [72]:
sales_by_engine = sales_moto['Engine_CC'].value_counts().reset_index(name='Total_Sales')
claims_by_engine = claims_moto['Engine_CC'].value_counts().reset_index(name='Total_Claims')

In [73]:
claim_rate_df = pd.merge(sales_by_engine, claims_by_engine, on='Engine_CC')
claim_rate_df['Claim_Rate_per_1000'] = (claim_rate_df['Total_Claims'] / claim_rate_df['Total_Sales']) * 1000

In [74]:
claim_rate_df.sort_values(by='Claim_Rate_per_1000', ascending=False)

,Engine_CC,Total_Sales,Total_Claims,Claim_Rate_per_1000
5,346,113,34,300.884956
6,535,87,26,298.850575
1,452,2424,624,257.425743
0,349,14344,3587,250.069716
2,648,1807,450,249.031544
3,411,817,197,241.126071
4,499,408,82,200.980392


In [75]:
claims['Claim_Amount'] = pd.to_numeric(claims['Claim_Amount'], errors='coerce')

In [76]:
approval_rate = claims.groupby('Component_Category')[['Claim_Amount', 'Approved_Amount']].sum()

In [77]:
approval_rate['Payout_Approval_Rate_Pct'] = (approval_rate['Approved_Amount'] / approval_rate['Claim_Amount']) * 100

In [78]:
approval_rate['Payout_Approval_Rate_Pct'].sort_values(ascending=False)

Component_Category
Fuel System & EFI        69.939191
Engine & Transmission    69.184025
Electrical System        69.105153
Frame & Chassis          68.036384
Exhaust System           67.465377
Suspension & Brakes      66.802535
Name: Payout_Approval_Rate_Pct, dtype: float64

### Challenge E

In [79]:
test_ride = pd.read_csv('TestRide.csv')
sales = pd.read_csv('Sales.csv')

In [80]:
overall_conv_rate = (test_ride['Converted_To_Sale'] == 'Yes').mean() * 100

In [81]:
completed_test_rides = test_ride[test_ride['Test_Ride_Status'] == 'Completed']
completed_conv_rate = (completed_test_rides['Converted_To_Sale'] == 'Yes').mean() * 100

In [84]:
overall_conv_rate

np.float64(42.0)

In [83]:
completed_conv_rate

np.float64(50.29337803855826)

In [85]:
converted = test_ride[test_ride['Converted_To_Sale'] == 'Yes'].copy()

In [86]:
converted = pd.merge(converted, sales[['Sale_ID', 'Sale_Date']], on='Sale_ID', how='inner')

In [87]:
converted['Test_Ride_Date'] = pd.to_datetime(converted['Test_Ride_Date'])
converted['Sale_Date'] = pd.to_datetime(converted['Sale_Date'])

In [88]:
converted['Conversion_Days'] = (converted['Sale_Date'] - converted['Test_Ride_Date']).dt.days

In [89]:
rating_speed = converted.groupby('Feedback_Rating')['Conversion_Days'].agg(['count', 'mean', 'median'])

In [90]:
rating_speed

,count,mean,median
Feedback_Rating,,,
4,599,7.666110,8.0
5,3601,7.611775,8.0


In [91]:
conv_by_rating = test_ride.groupby('Feedback_Rating')['Converted_To_Sale'].apply(lambda x: (x == 'Yes').mean() * 100)

In [92]:
conv_by_rating

Feedback_Rating
0     0.000000
1     0.000000
2     0.000000
3     0.000000
4    29.405989
5    74.033717
Name: Converted_To_Sale, dtype: float64